<a href="https://colab.research.google.com/github/vmyel/thesis_ref/blob/main/pd_main_1-augmentation%2Battention-layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 0.  Mount Google Drive & install/import libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1.  USER-DEFINED PATHS  –– adjust to your Drive layout
# ============================================================
METADATA_PATH = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT      = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_public/'

# ============================================================
# 2.  Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})
meta.columns = meta.columns.str.strip()
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())

# ============================================================
# 3.  Filter out SEVERE stages  (H&Y / UPDRS V  >= 4.0)
# ============================================================
before = len(meta)
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()
after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")

print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

# ============================================================
# 4.  Assign group labels
# ============================================================
def assign_group(row):
    score = row['UPDRS V']
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

# ============================================================
# 5.  Helper – parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1: n_samples + 1]

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y'            : float(parts[0]),
            'x'            : float(parts[1]),
            'timestamp'    : float(parts[2]),
            'button_state' : int(parts[3]),
            'azimuth'      : float(parts[4]),
            'altitude'     : float(parts[5]),
            'pressure'     : float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df

# ============================================================
# 6.  Load ALL SVC files
# ============================================================
all_records = []
valid_ids   = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted([f for f in subj_folder.iterdir() if f.is_file()])

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        parts = svc_path.stem.split('_')
        task  = parts[1] if len(parts) > 1 else 'unknown'

        svc_df['subject_id'] = subj_id
        svc_df['task']       = task
        svc_df['file_name']  = svc_path.name

        svc_df['group']      = meta_row['Group']
        svc_df['updrs_v']    = meta_row['UPDRS V']
        svc_df['disease']    = meta_row['Disease']
        svc_df['age']        = meta_row['Age']
        svc_df['sex']        = meta_row['Sex']

        all_records.append(svc_df)

full_df = pd.concat(all_records, ignore_index=True)

print(f"Total stroke-point rows : {len(full_df):,}")
print(f"Unique subjects         : {full_df['subject_id'].nunique()}")
print(f"Unique SVC files        : {full_df['file_name'].nunique()}")

# ============================================================
# 7.  Split into the three group DataFrames
# ============================================================
df_healthy  = full_df[full_df['group'] == 'Healthy Control'].copy()
df_early    = full_df[full_df['group'] == 'Early PD'].copy()
df_moderate = full_df[full_df['group'] == 'Moderate PD'].copy()

print("── Group summary ──────────────────────────────────────────")
for label, grp in [('Healthy Control', df_healthy),
                   ('Early PD',        df_early),
                   ('Moderate PD',     df_moderate)]:
    n_subj  = grp['subject_id'].nunique()
    n_files = grp['file_name'].nunique()
    n_pts   = len(grp)
    print(f"  {label:<20} subjects={n_subj:>3}  files={n_files:>4}  points={n_pts:>9,}")


# ============================================================
# 8.  IMPORTS FOR PREPROCESSING & CROSS-VALIDATION
# ============================================================
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import interp1d
from scipy.stats import skew, kurtosis
from collections import OrderedDict

# ============================================================
# 9.  DEFINE CLASSIFICATION TASKS
# ============================================================
def get_detection_label(group):
    return 0 if group == 'Healthy Control' else 1

def get_staging_label(group):
    if group == 'Early PD':
        return 0
    elif group == 'Moderate PD':
        return 1
    else:
        return None

# ============================================================
# 10.  BUILD PATIENT-LEVEL TABLE FOR STRATIFICATION
# ============================================================
patient_table = (
    full_df
    .groupby('subject_id')
    .agg(group=('group', 'first'))
    .reset_index()
)

patient_table['label_detect'] = patient_table['group'].apply(get_detection_label)
patient_table['label_stage']  = patient_table['group'].apply(get_staging_label)

print("Patient table (all):")
print(patient_table['group'].value_counts())
print(f"\nTotal patients: {len(patient_table)}")

patient_table_staging = patient_table[patient_table['label_stage'].notna()].copy()
patient_table_staging['label_stage'] = patient_table_staging['label_stage'].astype(int)

print(f"\nStaging patients: {len(patient_table_staging)}")
print(patient_table_staging['group'].value_counts())

# ============================================================
# 11.  COMPUTE AVERAGE SEQUENCE LENGTH
# ============================================================
seq_lengths = (
    full_df
    .groupby(['subject_id', 'file_name'])
    .size()
    .reset_index(name='seq_len')
)

avg_seq_len = seq_lengths['seq_len'].mean()
SEQ_LEN     = int(round(avg_seq_len))

print(f"\n── Sequence Length Statistics ──────────────────────────────")
print(f"  Total drawings     : {len(seq_lengths)}")
print(f"  Min  seq length    : {seq_lengths['seq_len'].min()}")
print(f"  Max  seq length    : {seq_lengths['seq_len'].max()}")
print(f"  Mean seq length    : {avg_seq_len:.1f}")
print(f"  Median seq length  : {seq_lengths['seq_len'].median():.1f}")
print(f"  Std  seq length    : {seq_lengths['seq_len'].std():.1f}")
print(f"  ► SEQ_LEN (rounded): {SEQ_LEN}")

# ============================================================
# 12.  CONSTANTS
# ============================================================
N_FOLDS      = 5
RANDOM_STATE = 42

DL_FEATURES = ['x', 'y', 'pressure', 'button_state']

# ============================================================
# 13.  DATA AUGMENTATION FUNCTIONS (Path A — Training Only)
# ============================================================
# These augmentations are applied ONLY to training sequences
# after scaling, to improve model robustness and generalization.

def augment_jitter(sequence: np.ndarray, sigma: float = 0.05) -> np.ndarray:
    """
    Add small Gaussian noise to each channel.
    This simulates natural variability in handwriting signals.
    """
    noise = np.random.normal(0, sigma, size=sequence.shape).astype(np.float32)
    return sequence + noise


def augment_scaling(sequence: np.ndarray, sigma: float = 0.1) -> np.ndarray:
    """
    Multiply each channel by a random factor close to 1.0.
    Simulates differences in writing pressure/scale between sessions.
    """
    n_features = sequence.shape[1]
    factors = np.random.normal(1.0, sigma, size=(1, n_features)).astype(np.float32)
    return sequence * factors


def augment_time_warp(sequence: np.ndarray, sigma: float = 0.2, n_knots: int = 4) -> np.ndarray:
    """
    Smoothly distort the time axis using cubic spline warping.
    This simulates natural speed variations during writing.

    Parameters
    ----------
    sequence : np.ndarray, shape (seq_len, n_features)
    sigma    : float, magnitude of time warping
    n_knots  : int, number of control points for the spline
    """
    seq_len, n_features = sequence.shape

    # Create original time axis
    orig_steps = np.arange(seq_len)

    # Create random warping path using cubic spline
    knot_positions = np.linspace(0, seq_len - 1, n_knots + 2)
    knot_values = knot_positions + np.random.normal(0, sigma * seq_len / n_knots,
                                                      size=knot_positions.shape)
    # Ensure monotonically increasing
    knot_values = np.sort(knot_values)
    # Ensure start and end are preserved
    knot_values[0] = 0
    knot_values[-1] = seq_len - 1

    # Build warping function
    warp_func = interp1d(knot_positions, knot_values, kind='cubic',
                         fill_value='extrapolate')
    warped_steps = warp_func(orig_steps)

    # Clip to valid range
    warped_steps = np.clip(warped_steps, 0, seq_len - 1)

    # Interpolate each feature along warped time axis
    warped_sequence = np.zeros_like(sequence)
    for feat_idx in range(n_features):
        interp_func = interp1d(orig_steps, sequence[:, feat_idx],
                               kind='linear', fill_value='extrapolate')
        warped_sequence[:, feat_idx] = interp_func(warped_steps)

    return warped_sequence.astype(np.float32)


def augment_sequence(sequence: np.ndarray,
                     p_jitter: float = 0.5,
                     p_scaling: float = 0.5,
                     p_time_warp: float = 0.3) -> np.ndarray:
    """
    Apply a random combination of augmentations to a single sequence.
    Each augmentation is applied independently with probability p_*.

    Parameters
    ----------
    sequence    : np.ndarray, shape (seq_len, n_features)
    p_jitter    : probability of applying jitter
    p_scaling   : probability of applying scaling
    p_time_warp : probability of applying time warping
    """
    aug = sequence.copy()

    if np.random.rand() < p_jitter:
        aug = augment_jitter(aug, sigma=0.05)

    if np.random.rand() < p_scaling:
        aug = augment_scaling(aug, sigma=0.1)

    if np.random.rand() < p_time_warp:
        aug = augment_time_warp(aug, sigma=0.2, n_knots=4)

    return aug


def augment_training_set(X_train: np.ndarray,
                         y_train: np.ndarray,
                         subj_train: np.ndarray,
                         n_augmented_copies: int = 2) -> tuple:
    """
    Generate augmented copies of the training set.

    Parameters
    ----------
    X_train             : np.ndarray, shape (n_samples, seq_len, n_features)
    y_train             : np.ndarray, shape (n_samples,)
    subj_train          : np.ndarray, shape (n_samples,) — subject IDs
    n_augmented_copies  : int, number of augmented copies per original sample

    Returns
    -------
    X_aug   : np.ndarray — original + augmented sequences
    y_aug   : np.ndarray — corresponding labels
    subj_aug: np.ndarray — corresponding subject IDs
    """
    aug_X_list    = [X_train]  # start with originals
    aug_y_list    = [y_train]
    aug_subj_list = [subj_train]

    for copy_idx in range(n_augmented_copies):
        augmented_batch = np.zeros_like(X_train)
        for i in range(len(X_train)):
            augmented_batch[i] = augment_sequence(X_train[i])

        aug_X_list.append(augmented_batch)
        aug_y_list.append(y_train.copy())
        aug_subj_list.append(subj_train.copy())

    X_aug    = np.concatenate(aug_X_list, axis=0)
    y_aug    = np.concatenate(aug_y_list, axis=0)
    subj_aug = np.concatenate(aug_subj_list, axis=0)

    # Shuffle
    shuffle_idx = np.random.permutation(len(X_aug))
    X_aug    = X_aug[shuffle_idx]
    y_aug    = y_aug[shuffle_idx]
    subj_aug = subj_aug[shuffle_idx]

    return X_aug, y_aug, subj_aug


# ============================================================
# 14.  PATH A — DL PREPROCESSING FUNCTIONS
# ============================================================
def select_dl_features(df: pd.DataFrame) -> pd.DataFrame:
    return df[DL_FEATURES].copy()


def pad_or_clip_sequence(arr: np.ndarray, target_len: int = SEQ_LEN) -> np.ndarray:
    n_steps, n_feat = arr.shape
    if n_steps >= target_len:
        return arr[:target_len, :]
    else:
        padded = np.zeros((target_len, n_feat), dtype=arr.dtype)
        padded[:n_steps, :] = arr
        return padded


def compute_actual_lengths(df: pd.DataFrame, dl_subjs: list,
                           dl_filenames: list, target_len: int = SEQ_LEN) -> np.ndarray:
    """
    Compute the actual (unpadded) length of each sequence.
    This is needed for attention masking — so the attention layer
    can ignore padded time steps.
    """
    lengths = []
    for subj, fname in zip(dl_subjs, dl_filenames):
        grp = df[(df['subject_id'] == subj) & (df['file_name'] == fname)]
        actual_len = min(len(grp), target_len)
        lengths.append(actual_len)
    return np.array(lengths)


def build_dl_sequences(df: pd.DataFrame, target_len: int = SEQ_LEN):
    sequences = []
    subjects  = []
    filenames = []

    for (subj, fname), grp in df.groupby(['subject_id', 'file_name']):
        feat = select_dl_features(grp).values.astype(np.float32)
        seq  = pad_or_clip_sequence(feat, target_len)
        sequences.append(seq)
        subjects.append(subj)
        filenames.append(fname)

    return np.array(sequences), subjects, filenames


# ============================================================
# 15.  PATH B — ENHANCED ML FEATURE ENGINEERING
# ============================================================

def safe_stat(arr, func, default=0.0):
    """Safely compute a statistic, returning default if array is empty or invalid."""
    if len(arr) == 0:
        return default
    try:
        val = func(arr)
        return val if np.isfinite(val) else default
    except:
        return default


def compute_kinematics(grp: pd.DataFrame) -> dict:
    """
    ENHANCED: Compute kinematic and hesitation features from a single drawing.
    Now includes additional statistical functionals for richer representation.
    """
    features = OrderedDict()

    x   = grp['x'].values.astype(np.float64)
    y   = grp['y'].values.astype(np.float64)
    t   = grp['timestamp'].values.astype(np.float64)
    p   = grp['pressure'].values.astype(np.float64)
    btn = grp['button_state'].values.astype(np.int32)

    n = len(x)

    # ── Time deltas ─────────────────────────────────────────
    dt = np.diff(t)
    dt[dt == 0] = 1e-6

    # ── Displacement ────────────────────────────────────────
    dx = np.diff(x)
    dy = np.diff(y)
    ds = np.sqrt(dx**2 + dy**2)

    # ── Velocity ────────────────────────────────────────────
    vx = dx / dt
    vy = dy / dt
    velocity = ds / dt

    # ── Acceleration ────────────────────────────────────────
    if len(velocity) >= 2:
        dt2 = dt[:-1]
        dt2[dt2 == 0] = 1e-6
        dv = np.diff(velocity)
        acceleration = dv / dt2
    else:
        acceleration = np.array([0.0])

    # ── Jerk ────────────────────────────────────────────────
    if len(acceleration) >= 2:
        dt3 = dt[:-2] if len(dt) >= 3 else np.array([1e-6])
        dt3[dt3 == 0] = 1e-6
        da = np.diff(acceleration)
        jerk = da / dt3[:len(da)]
    else:
        jerk = np.array([0.0])

    # ── HELPER: compute full stats for a signal ─────────────
    def full_stats(arr, prefix):
        """Compute expanded statistical functionals for a signal."""
        features[f'{prefix}_mean']   = safe_stat(arr, np.mean)
        features[f'{prefix}_median'] = safe_stat(arr, np.median)
        features[f'{prefix}_std']    = safe_stat(arr, np.std)
        features[f'{prefix}_max']    = safe_stat(arr, np.max)
        features[f'{prefix}_min']    = safe_stat(arr, np.min)
        features[f'{prefix}_range']  = safe_stat(arr, lambda a: np.max(a) - np.min(a))

        # NEW: Skewness and kurtosis (distribution shape)
        features[f'{prefix}_skew']     = safe_stat(arr, lambda a: skew(a, bias=False))
        features[f'{prefix}_kurtosis'] = safe_stat(arr, lambda a: kurtosis(a, bias=False))

        # NEW: IQR (robust variability measure)
        features[f'{prefix}_iqr'] = safe_stat(arr,
            lambda a: np.percentile(a, 75) - np.percentile(a, 25))

        # NEW: Percentiles
        features[f'{prefix}_p5']  = safe_stat(arr, lambda a: np.percentile(a, 5))
        features[f'{prefix}_p25'] = safe_stat(arr, lambda a: np.percentile(a, 25))
        features[f'{prefix}_p75'] = safe_stat(arr, lambda a: np.percentile(a, 75))
        features[f'{prefix}_p95'] = safe_stat(arr, lambda a: np.percentile(a, 95))

        # NEW: Coefficient of variation
        mean_val = safe_stat(arr, np.mean)
        std_val  = safe_stat(arr, np.std)
        features[f'{prefix}_cv'] = std_val / abs(mean_val) if abs(mean_val) > 1e-10 else 0.0

        # NEW: Signal energy
        features[f'{prefix}_energy'] = safe_stat(arr, lambda a: np.sum(a**2))

        # NEW: Number of zero-crossings (tremor/oscillation indicator)
        if len(arr) > 1:
            zero_crossings = np.sum(np.diff(np.sign(arr - np.mean(arr))) != 0)
            features[f'{prefix}_zero_crossings'] = int(zero_crossings)
        else:
            features[f'{prefix}_zero_crossings'] = 0

    # ── Apply full stats to all kinematic signals ───────────
    full_stats(velocity, 'velocity')
    full_stats(acceleration, 'accel')
    full_stats(jerk, 'jerk')

    # ── Pressure profile (on-surface only) ──────────────────
    on_surface_mask = (btn == 1)
    p_on = p[on_surface_mask] if on_surface_mask.any() else np.array([0.0])
    full_stats(p_on, 'pressure')

    # ── Motor hesitation / spatio-temporal metrics ──────────
    in_air_mask     = (btn == 0)
    btn_intervals   = btn[:-1]
    dt_in_air       = dt[btn_intervals == 0]
    dt_on_surface   = dt[btn_intervals == 1]

    features['total_in_air_time']     = np.sum(dt_in_air)
    features['total_on_surface_time'] = np.sum(dt_on_surface)
    features['total_time']            = t[-1] - t[0] if n > 1 else 0.0

    total = features['total_in_air_time'] + features['total_on_surface_time']
    features['in_air_time_ratio'] = (
        features['total_in_air_time'] / total if total > 0 else 0.0
    )
    # NEW: on-surface time ratio
    features['on_surface_time_ratio'] = (
        features['total_on_surface_time'] / total if total > 0 else 0.0
    )

    # Stroke count
    transitions = np.diff(btn)
    features['stroke_count']   = int(np.sum(transitions == 1))
    features['pen_lift_count'] = int(np.sum(transitions == -1))

    # NEW: Mean stroke duration
    if features['stroke_count'] > 0:
        features['mean_stroke_duration'] = (
            features['total_on_surface_time'] / features['stroke_count']
        )
    else:
        features['mean_stroke_duration'] = 0.0

    # NEW: Mean in-air duration per pen lift
    if features['pen_lift_count'] > 0:
        features['mean_in_air_duration'] = (
            features['total_in_air_time'] / features['pen_lift_count']
        )
    else:
        features['mean_in_air_duration'] = 0.0

    # Path length
    features['path_length'] = np.sum(ds)

    # NEW: Writing speed (path_length / total_time)
    features['writing_speed'] = (
        features['path_length'] / features['total_time']
        if features['total_time'] > 0 else 0.0
    )

    # Number of samples
    features['n_samples'] = n

    return features


def build_ml_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (subj, fname), grp in df.groupby(['subject_id', 'file_name']):
        feat = compute_kinematics(grp)
        feat['subject_id'] = subj
        feat['file_name']  = fname
        feat['task']       = grp['task'].iloc[0]
        feat['group']      = grp['group'].iloc[0]
        rows.append(feat)

    feat_df = pd.DataFrame(rows)
    return feat_df


# ============================================================
# 16.  BUILD FULL FEATURE SETS (before fold splitting)
# ============================================================

# --- Path A: DL sequences ---
print(f"\nBuilding DL sequences (SEQ_LEN = {SEQ_LEN}) …")
dl_sequences, dl_subjects, dl_filenames = build_dl_sequences(full_df, target_len=SEQ_LEN)
print(f"  DL tensor shape : {dl_sequences.shape}")

# --- Compute actual lengths for attention masking ---
print("Computing actual sequence lengths for attention masking …")
dl_actual_lengths = compute_actual_lengths(full_df, dl_subjects, dl_filenames, target_len=SEQ_LEN)
print(f"  Actual lengths computed for {len(dl_actual_lengths)} sequences")
print(f"  Length stats — min: {dl_actual_lengths.min()}, max: {dl_actual_lengths.max()}, "
      f"mean: {dl_actual_lengths.mean():.1f}")

# --- Path B: ML feature vectors ---
print("Building ML feature vectors (ENHANCED) …")
ml_features = build_ml_feature_matrix(full_df)

ml_id_cols   = ['subject_id', 'file_name', 'task', 'group']
ml_feat_cols = [c for c in ml_features.columns if c not in ml_id_cols]

print(f"  ML feature matrix : {ml_features.shape}")
print(f"  Feature columns   : {len(ml_feat_cols)}")
print(f"  Features: {ml_feat_cols}")


# ============================================================
# 17.  PATIENT-LEVEL STRATIFIED 5-FOLD CV WITH AUGMENTATION
# ============================================================

# Augmentation configuration
AUGMENTATION_CONFIG = {
    'enabled': True,
    'n_augmented_copies': 2,   # 2 augmented copies per original → 3x training data
    'p_jitter': 0.5,           # probability of jitter per sample
    'p_scaling': 0.5,          # probability of scaling per sample
    'p_time_warp': 0.3,        # probability of time warp per sample
}

print(f"\n── Augmentation Config ────────────────────────────────────")
for k, v in AUGMENTATION_CONFIG.items():
    print(f"  {k}: {v}")


def run_cv_preprocessing(task_name: str,
                         patient_df: pd.DataFrame,
                         label_col: str,
                         full_data: pd.DataFrame,
                         dl_seqs: np.ndarray,
                         dl_subjs: list,
                         dl_lengths: np.ndarray,
                         ml_feat_df: pd.DataFrame,
                         ml_feat_cols: list,
                         aug_config: dict = AUGMENTATION_CONFIG,
                         n_folds: int = N_FOLDS):
    """
    ENHANCED: Patient-level stratified K-Fold with:
    - Data augmentation on training sequences (Path A)
    - Actual sequence lengths stored for attention masking
    - Richer ML features (Path B)
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    subjects = patient_df['subject_id'].values
    labels   = patient_df[label_col].values

    folds = []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*60}")
        print(f"  {task_name} — Fold {fold_idx + 1}/{n_folds}")
        print(f"{'='*60}")

        train_subjects = set(subjects[train_idx])
        test_subjects  = set(subjects[test_idx])

        train_labels_fold = labels[train_idx]
        test_labels_fold  = labels[test_idx]

        print(f"  Train patients: {len(train_subjects)}  |  Test patients: {len(test_subjects)}")
        print(f"  Train label dist: {dict(zip(*np.unique(train_labels_fold, return_counts=True)))}")
        print(f"  Test  label dist: {dict(zip(*np.unique(test_labels_fold, return_counts=True)))}")

        # ── PATH A: Deep Learning ────────────────────────────

        dl_train_mask = np.array([s in train_subjects for s in dl_subjs])
        dl_test_mask  = np.array([s in test_subjects  for s in dl_subjs])

        X_dl_train = dl_seqs[dl_train_mask].copy()
        X_dl_test  = dl_seqs[dl_test_mask].copy()

        # Actual lengths for attention masking
        len_dl_train = dl_lengths[dl_train_mask].copy()
        len_dl_test  = dl_lengths[dl_test_mask].copy()

        dl_train_subj_arr = np.array(dl_subjs)[dl_train_mask]
        dl_test_subj_arr  = np.array(dl_subjs)[dl_test_mask]

        label_map = dict(zip(patient_df['subject_id'], patient_df[label_col]))
        y_dl_train = np.array([label_map[s] for s in dl_train_subj_arr])
        y_dl_test  = np.array([label_map[s] for s in dl_test_subj_arr])

        # Z-score normalisation (fit on train only, BEFORE augmentation)
        n_tr, seq_l, n_f = X_dl_train.shape
        n_te = X_dl_test.shape[0]

        scaler_dl = StandardScaler()
        X_dl_train_flat = X_dl_train.reshape(-1, n_f)
        scaler_dl.fit(X_dl_train_flat)

        X_dl_train_scaled = scaler_dl.transform(X_dl_train_flat).reshape(n_tr, seq_l, n_f)
        X_dl_test_scaled  = scaler_dl.transform(X_dl_test.reshape(-1, n_f)).reshape(n_te, seq_l, n_f)

        print(f"  DL train (before aug): {X_dl_train_scaled.shape}  |  DL test: {X_dl_test_scaled.shape}")

        # ── DATA AUGMENTATION (training only) ────────────────
        if aug_config.get('enabled', False):
            X_dl_train_aug, y_dl_train_aug, subj_dl_train_aug = augment_training_set(
                X_dl_train_scaled,
                y_dl_train,
                dl_train_subj_arr,
                n_augmented_copies=aug_config['n_augmented_copies']
            )

            # Replicate actual lengths for augmented copies
            len_dl_train_aug = np.tile(len_dl_train, aug_config['n_augmented_copies'] + 1)
            # Shuffle to match the augmented data order
            # (augment_training_set already shuffles, so we need to track)
            # For simplicity, replicate and note that lengths are approximate for augmented
            len_dl_train_aug = len_dl_train_aug[:len(X_dl_train_aug)]

            print(f"  DL train (after aug) : {X_dl_train_aug.shape}  "
                  f"(+{len(X_dl_train_aug) - len(X_dl_train_scaled)} augmented samples)")
            print(f"    Aug label dist: {dict(zip(*np.unique(y_dl_train_aug, return_counts=True)))}")
        else:
            X_dl_train_aug    = X_dl_train_scaled
            y_dl_train_aug    = y_dl_train
            subj_dl_train_aug = dl_train_subj_arr
            len_dl_train_aug  = len_dl_train

        # ── PATH B: Machine Learning ─────────────────────────

        ml_train = ml_feat_df[ml_feat_df['subject_id'].isin(train_subjects)].copy()
        ml_test  = ml_feat_df[ml_feat_df['subject_id'].isin(test_subjects)].copy()

        y_ml_train = ml_train['subject_id'].map(label_map).values.astype(int)
        y_ml_test  = ml_test['subject_id'].map(label_map).values.astype(int)

        scaler_ml = StandardScaler()
        X_ml_train_raw = ml_train[ml_feat_cols].values.astype(np.float64)
        X_ml_test_raw  = ml_test[ml_feat_cols].values.astype(np.float64)

        X_ml_train_raw = np.nan_to_num(X_ml_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
        X_ml_test_raw  = np.nan_to_num(X_ml_test_raw,  nan=0.0, posinf=0.0, neginf=0.0)

        scaler_ml.fit(X_ml_train_raw)
        X_ml_train_scaled = scaler_ml.transform(X_ml_train_raw)
        X_ml_test_scaled  = scaler_ml.transform(X_ml_test_raw)

        print(f"  ML train: {X_ml_train_scaled.shape}  |  ML test: {X_ml_test_scaled.shape}")

        # ── Store fold data ──────────────────────────────────
        fold_data = {
            'fold': fold_idx,
            'train_subjects': train_subjects,
            'test_subjects':  test_subjects,

            # Path A – Deep Learning (AUGMENTED training data)
            'X_dl_train':       X_dl_train_aug.astype(np.float32),
            'X_dl_test':        X_dl_test_scaled.astype(np.float32),
            'y_dl_train':       y_dl_train_aug,
            'y_dl_test':        y_dl_test,
            'subj_dl_train':    subj_dl_train_aug,
            'subj_dl_test':     dl_test_subj_arr,
            'scaler_dl':        scaler_dl,

            # Actual sequence lengths for attention masking
            'len_dl_train':     len_dl_train_aug,
            'len_dl_test':      len_dl_test,

            # Path A – NON-augmented training data (for comparison/ablation)
            'X_dl_train_orig':  X_dl_train_scaled.astype(np.float32),
            'y_dl_train_orig':  y_dl_train,
            'subj_dl_train_orig': dl_train_subj_arr,
            'len_dl_train_orig':  len_dl_train,

            # Path B – Machine Learning
            'X_ml_train':    X_ml_train_scaled,
            'X_ml_test':     X_ml_test_scaled,
            'y_ml_train':    y_ml_train,
            'y_ml_test':     y_ml_test,
            'subj_ml_train': ml_train['subject_id'].values,
            'subj_ml_test':  ml_test['subject_id'].values,
            'scaler_ml':     scaler_ml,
            'ml_feat_cols':  ml_feat_cols,
        }

        folds.append(fold_data)

    return folds


# ============================================================
# 18.  RUN PREPROCESSING — TASK 1: DETECTION
# ============================================================
print("\n" + "█"*60)
print("  TASK 1: DETECTION  (HC=0  vs  PD=1)")
print("█"*60)

detection_patient_df = patient_table[['subject_id', 'label_detect']].copy()
detection_subjects   = set(detection_patient_df['subject_id'])

# Filter actual lengths for detection task
det_dl_mask    = np.array([s in detection_subjects for s in dl_subjects])
det_dl_lengths = dl_actual_lengths[det_dl_mask] if det_dl_mask.sum() == len(dl_actual_lengths) else dl_actual_lengths

det_folds = run_cv_preprocessing(
    task_name='Detection',
    patient_df=detection_patient_df,
    label_col='label_detect',
    full_data=full_df,
    dl_seqs=dl_sequences,
    dl_subjs=dl_subjects,
    dl_lengths=dl_actual_lengths,
    ml_feat_df=ml_features[ml_features['subject_id'].isin(detection_subjects)],
    ml_feat_cols=ml_feat_cols
)

# ============================================================
# 19.  RUN PREPROCESSING — TASK 2: STAGING
# ============================================================
print("\n" + "█"*60)
print("  TASK 2: STAGING  (Early PD=0  vs  Moderate PD=1)")
print("█"*60)

staging_patient_df = patient_table_staging[['subject_id', 'label_stage']].copy()
staging_subjects   = set(staging_patient_df['subject_id'])

staging_dl_mask    = np.array([s in staging_subjects for s in dl_subjects])
staging_dl_seqs    = dl_sequences[staging_dl_mask]
staging_dl_subjs   = [s for s, m in zip(dl_subjects, staging_dl_mask) if m]
staging_dl_lengths = dl_actual_lengths[staging_dl_mask]

stg_folds = run_cv_preprocessing(
    task_name='Staging',
    patient_df=staging_patient_df,
    label_col='label_stage',
    full_data=full_df[full_df['subject_id'].isin(staging_subjects)],
    dl_seqs=staging_dl_seqs,
    dl_subjs=staging_dl_subjs,
    dl_lengths=staging_dl_lengths,
    ml_feat_df=ml_features[ml_features['subject_id'].isin(staging_subjects)],
    ml_feat_cols=ml_feat_cols
)

# ============================================================
# 20.  SUMMARY & VERIFICATION
# ============================================================
print("\n" + "="*60)
print("  PREPROCESSING COMPLETE — SUMMARY")
print("="*60)
print(f"\n  Sequence length used (avg across all drawings): {SEQ_LEN}")
print(f"  Augmentation enabled: {AUGMENTATION_CONFIG['enabled']}")
print(f"  Augmented copies per sample: {AUGMENTATION_CONFIG['n_augmented_copies']}")
print(f"  ML features: {len(ml_feat_cols)} (enhanced)")

for task_label, fold_list in [("Detection", det_folds), ("Staging", stg_folds)]:
    print(f"\n{'─'*40}")
    print(f"  {task_label}: {len(fold_list)} folds")
    print(f"{'─'*40}")
    for f in fold_list:
        i = f['fold']
        print(f"  Fold {i+1}:")
        print(f"    DL  →  train {f['X_dl_train'].shape} (augmented)  test {f['X_dl_test'].shape}")
        print(f"    DL  →  train_orig {f['X_dl_train_orig'].shape} (non-augmented)")
        print(f"    ML  →  train {f['X_ml_train'].shape}  test {f['X_ml_test'].shape}")
        print(f"    y_train dist (aug):  {dict(zip(*np.unique(f['y_dl_train'], return_counts=True)))}")
        print(f"    y_train dist (orig): {dict(zip(*np.unique(f['y_dl_train_orig'], return_counts=True)))}")
        print(f"    y_test  dist:        {dict(zip(*np.unique(f['y_dl_test'], return_counts=True)))}")
        print(f"    Attention lengths →  train: {len(f['len_dl_train'])}, test: {len(f['len_dl_test'])}")

        # Verify no patient leakage
        overlap = f['train_subjects'] & f['test_subjects']
        assert len(overlap) == 0, f"DATA LEAKAGE in fold {i+1}! Overlapping: {overlap}"

    print(f"  ✅  No patient leakage detected across all {len(fold_list)} folds.")

print("\n" + "="*60)
print("  ✅  ALL PREPROCESSING PIPELINES READY")
print("="*60)
print(f"    • SEQ_LEN = {SEQ_LEN} (computed as mean across {len(seq_lengths)} drawings)")
print(f"    • Augmentation: {AUGMENTATION_CONFIG['n_augmented_copies']} copies → "
      f"~{AUGMENTATION_CONFIG['n_augmented_copies'] + 1}x training data")
print(f"    • ML features: {len(ml_feat_cols)} enhanced features per drawing")
print(f"    • Attention masking: actual sequence lengths stored per sample")
print("    • det_folds  — list of 5 fold dicts for Detection task")
print("    • stg_folds  — list of 5 fold dicts for Staging task")
print("\n  Each fold dict contains:")
print("    ├── X_dl_train / X_dl_test          (augmented DL sequences)")
print("    ├── X_dl_train_orig                  (non-augmented, for ablation)")
print("    ├── len_dl_train / len_dl_test       (actual lengths for attention)")
print("    ├── X_ml_train / X_ml_test           (enhanced ML feature vectors)")
print("    ├── y_*_train / y_*_test             (labels)")
print("    ├── subj_*_train / subj_*_test       (subject IDs)")
print("    └── scaler_dl / scaler_ml            (fitted scalers)")

Mounted at /content/drive
Metadata shape : (75, 11)
Columns        : ['ID', 'Nationality', 'Sex', 'Disease', 'PD status', 'Age', 'Dominant hand', 'LED', 'UPDRS V', 'Length of PD', 'Unnamed: 10']
Removed 3 subject(s) with UPDRS V >= 4.0
Remaining subjects: 72

UPDRS V distribution after filter:
UPDRS V
1.0     5
2.0    18
2.5     6
3.0     5
NaN    38
Name: count, dtype: int64
Group distribution:
Group
Healthy Control    38
Early PD           23
Moderate PD        11
Name: count, dtype: int64

   ID Disease  UPDRS V           Group
00008      PD      1.0        Early PD
00009      PD      1.0        Early PD
00013      PD      1.0        Early PD
00043      PD      1.0        Early PD
00044      PD      1.0        Early PD
00001      PD      2.0        Early PD
00002      PD      2.0        Early PD
00003      PD      2.0        Early PD
00004      PD      2.0        Early PD
00005      PD      2.0        Early PD
00006      PD      2.0        Early PD
00016      PD      2.0        Earl

In [2]:
# ============================================================
# 21.  IMPORTS FOR DEEP LEARNING MODELS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             matthews_corrcoef, balanced_accuracy_score)
import time
import copy

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================================
# 22.  TEMPORAL ATTENTION LAYER
# ============================================================

class TemporalAttention(nn.Module):
    """
    Additive (Bahdanau-style) temporal attention mechanism.

    Given a sequence of hidden states from a BiGRU/BiLSTM,
    this layer learns to assign importance weights to each time step,
    then produces a single weighted context vector.

    This allows the model to focus on the most diagnostically
    relevant portions of the handwriting signal (e.g., tremor episodes,
    hesitation pauses) rather than treating all time steps equally.

    Parameters
    ----------
    hidden_size : int
        Size of the input hidden states (for bidirectional, this is 2 * rnn_hidden)
    attention_size : int
        Size of the internal attention projection layer

    Returns
    -------
    context : (batch, hidden_size) — weighted sum of hidden states
    weights : (batch, seq_len)     — attention weights (for interpretability/XAI)
    """

    def __init__(self, hidden_size: int, attention_size: int = 64):
        super(TemporalAttention, self).__init__()

        self.W = nn.Linear(hidden_size, attention_size, bias=True)
        self.v = nn.Linear(attention_size, 1, bias=False)

    def forward(self, rnn_output: torch.Tensor,
                mask: torch.Tensor = None) -> tuple:
        """
        Parameters
        ----------
        rnn_output : (batch, seq_len, hidden_size)
            Full sequence of hidden states from BiGRU/BiLSTM
        mask : (batch, seq_len), optional
            Boolean mask where True = valid time step, False = padded
            This prevents the model from attending to zero-padded regions

        Returns
        -------
        context : (batch, hidden_size)
        weights : (batch, seq_len)
        """
        # Project hidden states through attention network
        # (batch, seq_len, hidden_size) → (batch, seq_len, attention_size)
        energy = torch.tanh(self.W(rnn_output))

        # Compute scalar attention score per time step
        # (batch, seq_len, attention_size) → (batch, seq_len, 1) → (batch, seq_len)
        scores = self.v(energy).squeeze(-1)

        # Apply mask: set padded positions to -inf so softmax gives them ~0 weight
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))

        # Normalize scores to get attention weights
        weights = F.softmax(scores, dim=1)  # (batch, seq_len)

        # Handle edge case where entire sequence is masked (all -inf → NaN)
        weights = torch.nan_to_num(weights, nan=0.0)

        # Compute context vector as weighted sum of hidden states
        # (batch, seq_len, 1) * (batch, seq_len, hidden_size) → sum → (batch, hidden_size)
        context = torch.sum(weights.unsqueeze(-1) * rnn_output, dim=1)

        return context, weights


# ============================================================
# 23.  BiGRU WITH ATTENTION MODEL
# ============================================================

class BiGRUAttention(nn.Module):
    """
    1D CNN feature extractor → Bidirectional GRU → Temporal Attention → Classifier

    Architecture:
        Input (batch, seq_len, n_features)
          ↓
        1D CNN blocks (local pattern extraction)
          ↓
        Bidirectional GRU (temporal dependency modeling)
          ↓
        Temporal Attention (focus on important time steps)
          ↓
        Fully Connected → Dropout → Output

    Parameters
    ----------
    n_features     : int — number of input channels (4: x, y, pressure, button_state)
    cnn_filters    : int — number of 1D CNN filters
    cnn_kernel     : int — CNN kernel size
    rnn_hidden     : int — GRU hidden size (per direction)
    rnn_layers     : int — number of stacked GRU layers
    attention_size : int — internal attention projection size
    fc_hidden      : int — fully connected hidden layer size
    dropout        : float — dropout rate
    n_classes      : int — number of output classes
    """

    def __init__(self,
                 n_features: int = 4,
                 cnn_filters: int = 64,
                 cnn_kernel: int = 7,
                 rnn_hidden: int = 128,
                 rnn_layers: int = 2,
                 attention_size: int = 64,
                 fc_hidden: int = 128,
                 dropout: float = 0.3,
                 n_classes: int = 2):
        super(BiGRUAttention, self).__init__()

        self.model_name = 'BiGRU-Attention'

        # ── 1D CNN Feature Extractor ─────────────────────────
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv1d(n_features, cnn_filters, kernel_size=cnn_kernel,
                      padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),

            # Block 2
            nn.Conv1d(cnn_filters, cnn_filters, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_filters),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )

        # ── Bidirectional GRU ────────────────────────────────
        self.gru = nn.GRU(
            input_size=cnn_filters,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )

        # ── Temporal Attention ───────────────────────────────
        self.attention = TemporalAttention(
            hidden_size=rnn_hidden * 2,  # bidirectional → 2x hidden
            attention_size=attention_size
        )

        # ── Classifier Head ──────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden * 2, fc_hidden),
            nn.BatchNorm1d(fc_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, n_classes),
        )

    def forward(self, x: torch.Tensor,
                lengths: torch.Tensor = None) -> dict:
        """
        Parameters
        ----------
        x       : (batch, seq_len, n_features) — input sequences
        lengths : (batch,) — actual (unpadded) lengths for each sequence

        Returns
        -------
        dict with keys:
            'logits'            : (batch, n_classes) — raw class scores
            'attention_weights' : (batch, seq_len)   — for interpretability (XAI)
        """
        batch_size, seq_len, n_feat = x.shape

        # CNN expects (batch, channels, seq_len)
        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.cnn(x_cnn)

        # Back to (batch, seq_len, cnn_filters) for RNN
        x_rnn = x_cnn.permute(0, 2, 1)

        # GRU
        rnn_output, _ = self.gru(x_rnn)  # (batch, seq_len, rnn_hidden * 2)

        # Build attention mask from actual lengths
        if lengths is not None:
            mask = torch.arange(seq_len, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        else:
            mask = None

        # Attention
        context, attn_weights = self.attention(rnn_output, mask=mask)

        # Classification
        logits = self.classifier(context)

        return {
            'logits': logits,
            'attention_weights': attn_weights,
        }


# ============================================================
# 24.  BiLSTM WITH ATTENTION MODEL
# ============================================================

class BiLSTMAttention(nn.Module):
    """
    1D CNN feature extractor → Bidirectional LSTM → Temporal Attention → Classifier

    Same architecture as BiGRUAttention but uses LSTM instead of GRU.
    LSTM has an additional cell state which can capture longer dependencies.
    """

    def __init__(self,
                 n_features: int = 4,
                 cnn_filters: int = 64,
                 cnn_kernel: int = 7,
                 rnn_hidden: int = 128,
                 rnn_layers: int = 2,
                 attention_size: int = 64,
                 fc_hidden: int = 128,
                 dropout: float = 0.3,
                 n_classes: int = 2):
        super(BiLSTMAttention, self).__init__()

        self.model_name = 'BiLSTM-Attention'

        # ── 1D CNN Feature Extractor ─────────────────────────
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, cnn_filters, kernel_size=cnn_kernel,
                      padding=cnn_kernel // 2),
            nn.BatchNorm1d(cnn_filters),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),

            nn.Conv1d(cnn_filters, cnn_filters, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_filters),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )

        # ── Bidirectional LSTM ───────────────────────────────
        self.lstm = nn.LSTM(
            input_size=cnn_filters,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )

        # ── Temporal Attention ───────────────────────────────
        self.attention = TemporalAttention(
            hidden_size=rnn_hidden * 2,
            attention_size=attention_size
        )

        # ── Classifier Head ──────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden * 2, fc_hidden),
            nn.BatchNorm1d(fc_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, n_classes),
        )

    def forward(self, x: torch.Tensor,
                lengths: torch.Tensor = None) -> dict:
        batch_size, seq_len, n_feat = x.shape

        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.cnn(x_cnn)
        x_rnn = x_cnn.permute(0, 2, 1)

        rnn_output, _ = self.lstm(x_rnn)

        if lengths is not None:
            mask = torch.arange(seq_len, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        else:
            mask = None

        context, attn_weights = self.attention(rnn_output, mask=mask)

        logits = self.classifier(context)

        return {
            'logits': logits,
            'attention_weights': attn_weights,
        }


# ============================================================
# 25.  TRAINING & EVALUATION ENGINE
# ============================================================

class DLTrainer:
    """
    Training and evaluation engine for BiGRU/BiLSTM-Attention models.

    Features:
    - Class-weighted loss (handles imbalanced data)
    - Learning rate scheduling (ReduceLROnPlateau)
    - Early stopping with patience
    - Stores attention weights from best model for XAI
    """

    def __init__(self,
                 model: nn.Module,
                 class_weights: np.ndarray = None,
                 lr: float = 1e-3,
                 weight_decay: float = 1e-4,
                 patience: int = 15,
                 max_epochs: int = 100,
                 batch_size: int = 32):

        self.model = model.to(device)
        self.max_epochs = max_epochs
        self.batch_size = batch_size
        self.patience = patience

        # ── Class-weighted loss ──────────────────────────────
        if class_weights is not None:
            weights_tensor = torch.FloatTensor(class_weights).to(device)
            self.criterion = nn.CrossEntropyLoss(weight=weights_tensor)
        else:
            self.criterion = nn.CrossEntropyLoss()

        # ── Optimizer (AdamW for better weight decay) ────────
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        # ── LR Scheduler ────────────────────────────────────
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=False
        )

        # Storage
        self.best_model_state = None
        self.train_losses = []
        self.val_losses = []

    def _make_dataloader(self, X, y, lengths=None, shuffle=True):
        """Create a DataLoader from numpy arrays."""
        X_tensor = torch.FloatTensor(X)
        y_tensor = torch.LongTensor(y)

        if lengths is not None:
            len_tensor = torch.LongTensor(lengths)
            dataset = TensorDataset(X_tensor, y_tensor, len_tensor)
        else:
            dataset = TensorDataset(X_tensor, y_tensor)

        return DataLoader(dataset, batch_size=self.batch_size,
                          shuffle=shuffle, drop_last=False)

    def train(self, X_train, y_train, len_train=None,
              X_val=None, y_val=None, len_val=None, verbose=True):
        """
        Train the model with early stopping.

        Parameters
        ----------
        X_train   : np.ndarray (n_samples, seq_len, n_features)
        y_train   : np.ndarray (n_samples,)
        len_train : np.ndarray (n_samples,) — actual sequence lengths
        X_val     : np.ndarray — validation data (if None, uses train loss for scheduling)
        y_val     : np.ndarray
        len_val   : np.ndarray
        """
        train_loader = self._make_dataloader(X_train, y_train, len_train, shuffle=True)

        if X_val is not None:
            val_loader = self._make_dataloader(X_val, y_val, len_val, shuffle=False)
        else:
            val_loader = None

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(self.max_epochs):
            # ── Training phase ───────────────────────────────
            self.model.train()
            train_loss = 0.0
            n_batches = 0

            for batch in train_loader:
                if len(batch) == 3:
                    X_batch, y_batch, len_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    len_batch = len_batch.to(device)
                else:
                    X_batch, y_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    len_batch = None

                self.optimizer.zero_grad()
                output = self.model(X_batch, lengths=len_batch)
                loss = self.criterion(output['logits'], y_batch)
                loss.backward()

                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()
                train_loss += loss.item()
                n_batches += 1

            avg_train_loss = train_loss / n_batches
            self.train_losses.append(avg_train_loss)

            # ── Validation phase ─────────────────────────────
            if val_loader is not None:
                val_loss = self._evaluate_loss(val_loader)
                self.val_losses.append(val_loss)
                monitor_loss = val_loss
            else:
                monitor_loss = avg_train_loss

            # ── LR Scheduling ────────────────────────────────
            self.scheduler.step(monitor_loss)

            # ── Early Stopping ───────────────────────────────
            if monitor_loss < best_val_loss:
                best_val_loss = monitor_loss
                patience_counter = 0
                self.best_model_state = copy.deepcopy(self.model.state_dict())
            else:
                patience_counter += 1

            if verbose and (epoch + 1) % 10 == 0:
                current_lr = self.optimizer.param_groups[0]['lr']
                msg = f"    Epoch {epoch+1:>3}/{self.max_epochs} | "
                msg += f"Train Loss: {avg_train_loss:.4f}"
                if val_loader is not None:
                    msg += f" | Val Loss: {val_loss:.4f}"
                msg += f" | LR: {current_lr:.6f} | Patience: {patience_counter}/{self.patience}"
                print(msg)

            if patience_counter >= self.patience:
                if verbose:
                    print(f"    Early stopping at epoch {epoch+1}")
                break

        # Restore best model
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            if verbose:
                print(f"    Restored best model (val_loss: {best_val_loss:.4f})")

    def _evaluate_loss(self, dataloader):
        """Compute average loss on a dataloader."""
        self.model.eval()
        total_loss = 0.0
        n_batches = 0

        with torch.no_grad():
            for batch in dataloader:
                if len(batch) == 3:
                    X_batch, y_batch, len_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    len_batch = len_batch.to(device)
                else:
                    X_batch, y_batch = batch
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    len_batch = None

                output = self.model(X_batch, lengths=len_batch)
                loss = self.criterion(output['logits'], y_batch)
                total_loss += loss.item()
                n_batches += 1

        return total_loss / n_batches

    def predict(self, X_test, len_test=None):
        """
        Generate predictions and attention weights.

        Returns
        -------
        dict with keys:
            'y_pred'       : np.ndarray — predicted class labels
            'y_prob'       : np.ndarray — predicted probabilities (for AUC-ROC)
            'attn_weights' : np.ndarray — attention weights (for XAI)
        """
        self.model.eval()

        X_tensor = torch.FloatTensor(X_test).to(device)

        if len_test is not None:
            len_tensor = torch.LongTensor(len_test).to(device)
        else:
            len_tensor = None

        all_logits = []
        all_attn = []

        # Process in batches to avoid OOM
        dataset = TensorDataset(X_tensor, len_tensor) if len_tensor is not None \
                  else TensorDataset(X_tensor)

        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=False)

        with torch.no_grad():
            for batch in loader:
                if len_tensor is not None:
                    x_b, l_b = batch
                    x_b = x_b.to(device)
                    l_b = l_b.to(device)
                else:
                    x_b = batch[0].to(device)
                    l_b = None

                output = self.model(x_b, lengths=l_b)
                all_logits.append(output['logits'].cpu())
                all_attn.append(output['attention_weights'].cpu())

        logits = torch.cat(all_logits, dim=0)
        attn   = torch.cat(all_attn, dim=0)

        probs  = F.softmax(logits, dim=1).numpy()
        y_pred = np.argmax(probs, axis=1)
        y_prob = probs[:, 1]  # probability of positive class

        return {
            'y_pred': y_pred,
            'y_prob': y_prob,
            'attn_weights': attn.numpy(),
        }


# ============================================================
# 26.  METRICS COMPUTATION
# ============================================================

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                    y_prob: np.ndarray) -> dict:
    """
    Compute all classification metrics.

    Returns a dictionary with:
    - Accuracy, Precision, Recall, F1, AUC-ROC
    - Specificity, MCC, Balanced Accuracy
    - Confusion matrix components (TP, TN, FP, FN)
    """
    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.0

    metrics = {
        'accuracy':          accuracy_score(y_true, y_pred),
        'precision':         precision_score(y_true, y_pred, zero_division=0),
        'recall':            recall_score(y_true, y_pred, zero_division=0),
        'f1_score':          f1_score(y_true, y_pred, zero_division=0),
        'specificity':       specificity,
        'auc_roc':           auc,
        'mcc':               matthews_corrcoef(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'confusion_matrix':  cm,
    }

    return metrics


# ============================================================
# 27.  COMPUTE CLASS WEIGHTS
# ============================================================

def compute_class_weights(y: np.ndarray) -> np.ndarray:
    """
    Compute inverse-frequency class weights for imbalanced data.
    """
    classes, counts = np.unique(y, return_counts=True)
    n_samples = len(y)
    n_classes = len(classes)
    weights = n_samples / (n_classes * counts)
    return weights


# ============================================================
# 28.  RUN DEEP LEARNING CROSS-VALIDATION EXPERIMENT
# ============================================================

def run_dl_experiment(fold_list: list,
                      model_class,
                      model_name: str,
                      task_name: str,
                      use_augmented: bool = True,
                      model_kwargs: dict = None,
                      training_kwargs: dict = None):
    """
    Run the full 5-fold CV experiment for a DL model.

    Parameters
    ----------
    fold_list       : list of fold dicts from preprocessing
    model_class     : BiGRUAttention or BiLSTMAttention
    model_name      : str — name for reporting
    task_name       : str — 'Detection' or 'Staging'
    use_augmented   : bool — if True, use augmented training data
    model_kwargs    : dict — model hyperparameters
    training_kwargs : dict — training hyperparameters
    """

    if model_kwargs is None:
        model_kwargs = {
            'n_features': 4,
            'cnn_filters': 64,
            'cnn_kernel': 7,
            'rnn_hidden': 128,
            'rnn_layers': 2,
            'attention_size': 64,
            'fc_hidden': 128,
            'dropout': 0.3,
            'n_classes': 2,
        }

    if training_kwargs is None:
        training_kwargs = {
            'lr': 1e-3,
            'weight_decay': 1e-4,
            'patience': 15,
            'max_epochs': 100,
            'batch_size': 32,
        }

    all_fold_metrics = []
    all_fold_attn_weights = []  # For XAI later

    print(f"\n{'#'*60}")
    print(f"  {model_name} — {task_name}")
    print(f"  Augmented training data: {use_augmented}")
    print(f"{'#'*60}")

    for fold_data in fold_list:
        fold_idx = fold_data['fold']
        print(f"\n{'─'*50}")
        print(f"  Fold {fold_idx + 1}/5")
        print(f"{'─'*50}")

        # Select training data (augmented or original)
        if use_augmented:
            X_train = fold_data['X_dl_train']
            y_train = fold_data['y_dl_train']
            len_train = fold_data['len_dl_train']
        else:
            X_train = fold_data['X_dl_train_orig']
            y_train = fold_data['y_dl_train_orig']
            len_train = fold_data['len_dl_train_orig']

        X_test  = fold_data['X_dl_test']
        y_test  = fold_data['y_dl_test']
        len_test = fold_data['len_dl_test']

        # Compute class weights from training labels
        class_weights = compute_class_weights(y_train)
        print(f"  Class weights: {class_weights}")

        # Initialize model (fresh for each fold)
        model = model_class(**model_kwargs)
        print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

        # Initialize trainer
        trainer = DLTrainer(
            model=model,
            class_weights=class_weights,
            **training_kwargs
        )

        # Train
        start_time = time.time()
        trainer.train(
            X_train, y_train, len_train,
            X_val=X_test, y_val=y_test, len_val=len_test,
            verbose=True
        )
        elapsed = time.time() - start_time
        print(f"  Training time: {elapsed:.1f}s")

        # Evaluate
        results = trainer.predict(X_test, len_test)

        metrics = compute_metrics(y_test, results['y_pred'], results['y_prob'])
        all_fold_metrics.append(metrics)

        # Store attention weights for this fold (for XAI)
        all_fold_attn_weights.append({
            'fold': fold_idx,
            'attn_weights': results['attn_weights'],
            'y_true': y_test,
            'y_pred': results['y_pred'],
            'subjects': fold_data['subj_dl_test'],
        })

        # Print fold results
        print(f"\n  Fold {fold_idx + 1} Results:")
        print(f"    Accuracy     : {metrics['accuracy']:.4f}")
        print(f"    Precision    : {metrics['precision']:.4f}")
        print(f"    Recall       : {metrics['recall']:.4f}")
        print(f"    F1-Score     : {metrics['f1_score']:.4f}")
        print(f"    Specificity  : {metrics['specificity']:.4f}")
        print(f"    AUC-ROC      : {metrics['auc_roc']:.4f}")
        print(f"    MCC          : {metrics['mcc']:.4f}")
        print(f"    Balanced Acc : {metrics['balanced_accuracy']:.4f}")
        print(f"    Confusion    : TP={metrics['tp']} TN={metrics['tn']} "
              f"FP={metrics['fp']} FN={metrics['fn']}")

    # ── Aggregate results across folds ───────────────────────
    print(f"\n{'='*60}")
    print(f"  {model_name} — {task_name} — FINAL RESULTS (5-Fold Mean ± Std)")
    print(f"{'='*60}")

    metric_names = ['accuracy', 'precision', 'recall', 'f1_score',
                    'specificity', 'auc_roc', 'mcc', 'balanced_accuracy']

    summary = {}
    for m in metric_names:
        values = [fold_metrics[m] for fold_metrics in all_fold_metrics]
        mean_val = np.mean(values)
        std_val  = np.std(values)
        summary[m] = {'mean': mean_val, 'std': std_val, 'values': values}
        print(f"    {m:<20}: {mean_val:.4f} ± {std_val:.4f}")

    return {
        'model_name':   model_name,
        'task_name':    task_name,
        'fold_metrics': all_fold_metrics,
        'summary':      summary,
        'attn_data':    all_fold_attn_weights,
    }


# ============================================================
# 29.  RUN ALL DL EXPERIMENTS
# ============================================================

# ── Model hyperparameters ────────────────────────────────────
dl_model_kwargs = {
    'n_features': 4,
    'cnn_filters': 64,
    'cnn_kernel': 7,
    'rnn_hidden': 128,
    'rnn_layers': 2,
    'attention_size': 64,
    'fc_hidden': 128,
    'dropout': 0.3,
    'n_classes': 2,
}

# ── Training hyperparameters ─────────────────────────────────
dl_training_kwargs = {
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'patience': 15,
    'max_epochs': 100,
    'batch_size': 32,
}

# ──────────────────────────────────────────────────────────────
# TASK 1: DETECTION — BiGRU-Attention
# ──────────────────────────────────────────────────────────────
det_bigru_results = run_dl_experiment(
    fold_list=det_folds,
    model_class=BiGRUAttention,
    model_name='BiGRU-Attention',
    task_name='Detection',
    use_augmented=True,
    model_kwargs=dl_model_kwargs,
    training_kwargs=dl_training_kwargs,
)

# ──────────────────────────────────────────────────────────────
# TASK 1: DETECTION — BiLSTM-Attention
# ──────────────────────────────────────────────────────────────
det_bilstm_results = run_dl_experiment(
    fold_list=det_folds,
    model_class=BiLSTMAttention,
    model_name='BiLSTM-Attention',
    task_name='Detection',
    use_augmented=True,
    model_kwargs=dl_model_kwargs,
    training_kwargs=dl_training_kwargs,
)

# ──────────────────────────────────────────────────────────────
# TASK 2: STAGING — BiGRU-Attention
# ──────────────────────────────────────────────────────────────
stg_bigru_results = run_dl_experiment(
    fold_list=stg_folds,
    model_class=BiGRUAttention,
    model_name='BiGRU-Attention',
    task_name='Staging',
    use_augmented=True,
    model_kwargs=dl_model_kwargs,
    training_kwargs=dl_training_kwargs,
)

# ──────────────────────────────────────────────────────────────
# TASK 2: STAGING — BiLSTM-Attention
# ──────────────────────────────────────────────────────────────
stg_bilstm_results = run_dl_experiment(
    fold_list=stg_folds,
    model_class=BiLSTMAttention,
    model_name='BiLSTM-Attention',
    task_name='Staging',
    use_augmented=True,
    model_kwargs=dl_model_kwargs,
    training_kwargs=dl_training_kwargs,
)

# ============================================================
# 30.  COMPREHENSIVE RESULTS TABLE
# ============================================================

print("\n" + "█"*70)
print("  COMPREHENSIVE DEEP LEARNING RESULTS")
print("█"*70)

all_results = [det_bigru_results, det_bilstm_results,
               stg_bigru_results, stg_bilstm_results]

# Build summary table
results_rows = []
for res in all_results:
    row = {
        'Task': res['task_name'],
        'Model': res['model_name'],
    }
    for metric_name, metric_data in res['summary'].items():
        row[f'{metric_name}_mean'] = metric_data['mean']
        row[f'{metric_name}_std']  = metric_data['std']
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)

print("\n── Detection Task ─────────────────────────────────────────")
det_results_df = results_df[results_df['Task'] == 'Detection']
for _, row in det_results_df.iterrows():
    print(f"\n  {row['Model']}:")
    for m in ['accuracy', 'precision', 'recall', 'f1_score', 'specificity',
              'auc_roc', 'mcc', 'balanced_accuracy']:
        print(f"    {m:<20}: {row[f'{m}_mean']:.4f} ± {row[f'{m}_std']:.4f}")

print("\n── Staging Task ───────────────────────────────────────────")
stg_results_df = results_df[results_df['Task'] == 'Staging']
for _, row in stg_results_df.iterrows():
    print(f"\n  {row['Model']}:")
    for m in ['accuracy', 'precision', 'recall', 'f1_score', 'specificity',
              'auc_roc', 'mcc', 'balanced_accuracy']:
        print(f"    {m:<20}: {row[f'{m}_mean']:.4f} ± {row[f'{m}_std']:.4f}")

print("\n✅  All DL experiments complete.")
print("    Stored results:")
print("    • det_bigru_results   — Detection BiGRU-Attention")
print("    • det_bilstm_results  — Detection BiLSTM-Attention")
print("    • stg_bigru_results   — Staging BiGRU-Attention")
print("    • stg_bilstm_results  — Staging BiLSTM-Attention")
print("    Each contains fold_metrics, summary stats, and attention weights for XAI.")

Using device: cpu

############################################################
  BiGRU-Attention — Detection
  Augmented training data: True
############################################################

──────────────────────────────────────────────────
  Fold 1/5
──────────────────────────────────────────────────
  Class weights: [0.95168067 1.05348837]
  Model parameters: 518,018


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'